# Using Project Source Code

This chapter demonstrates how to import library code from `src/` without
`sys.path` hacks. The project is installed in editable mode when you run
`uv sync`, so notebooks can import packages directly.

The example pipeline:

1. Generate synthetic data with a small simulation
2. Bundle a Random Forest and an SVM (with `StandardScaler`) into a single `MultiRegressor` composite estimator
3. Hand that composite to `compare_models`, which fits each base, computes bootstrap confidence intervals, MAPIE split conformal prediction intervals, and bootstrap-CI'd regression metrics
4. Visualize the comparison with faceted Altair charts

## Imports

No path bootstrapping is required — the packages are installed by `uv sync`.

In [3]:
from sklearn.model_selection import train_test_split

from analysis import compare_models
from core import ModelKind, Settings, TrainingData, build_split_dataset
from prediction import (
    MultiRegressor,
    random_forest_regressor,
    regression_pipeline,
    svm_regressor,
)
from simulation import generate_dataset
from visualization import (
    plot_dataset,
    plot_interval_metrics,
    plot_intervals,
    plot_regression_metrics,
)

## Generate synthetic data

In [ ]:
settings = Settings(n_samples=5000, seed=0, svm_gamma=0.025)
data = generate_dataset(settings)
data.head()

,x,y
0,2.739234,6.373780
1,-4.604266,-12.387189
2,-9.180530,-13.231512
3,-9.669447,-11.510018
4,6.265405,-2.772410


In [5]:
plot_dataset(data)

alt.LayerChart(...)

## Split into train, calibration, and test sets

In [6]:
train, remainder = train_test_split(data, test_size=0.3, random_state=0)
calib, test = train_test_split(remainder, test_size=0.5, random_state=0)
split_data = build_split_dataset(
    TrainingData.validate(train),
    TrainingData.validate(calib),
    TrainingData.validate(test),
)
len(train), len(calib), len(test)

(1050, 225, 225)

## Build the composite estimator

Each model is wired through `regression_pipeline(...)` to inherit the polynomial + Fourier feature
expansion. The SVM factory additionally wraps `SVR` behind a `StandardScaler` (an inner pipeline),
so scaling happens after feature expansion and only for the model that needs it.

`MultiRegressor` is a sklearn-native composite (`BaseEstimator + RegressorMixin + TransformerMixin`).
`.transform(X)` returns per-base predictions as a `(n_samples, n_estimators)` matrix, and the unfitted
estimator specs stay introspectable via `.estimators` — which is what `bootstrap_confidence_intervals`
and `fit_conformal` need to `clone()` per base.

In [7]:
random_forest_pipeline = regression_pipeline(random_forest_regressor(settings), settings)
random_forest_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('polynomialfeatures', ...), ('fourierfeatures', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",5
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a 

In [8]:
svm_pipeline = regression_pipeline(svm_regressor(settings), settings)
svm_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('polynomialfeatures', ...), ('fourierfeatures', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",5
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a 

In [9]:
regressors = MultiRegressor(
    estimators=[
        (ModelKind.RANDOM_FOREST.value, random_forest_pipeline),
        (ModelKind.SVM.value, svm_pipeline),
    ],
)
regressors

,estimators,"[('random_forest', ...), ('svm', ...)]"
Name,Type,Value
estimators_,list,[]
named_estimators_,dict,{}
names_,list,"['ra...st', 'svm']"
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('polynomialfeatures', ...), ('fourierfeatures', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",5


## Fit, calibrate, score, and bootstrap in one call

`compare_models` walks each `(name, pipeline)` pair in the composite once and returns a
`ModelComparisonReport` dataclass with six tagged DataFrames:

- `predictions` — per-model point predictions with ground truth
- `confidence` — bootstrap confidence intervals (refit-on-resample) per model
- `prediction` — MAPIE split conformal prediction intervals per model
- `regression_metrics` — RMSE/MAE/R² with bootstrap CIs, per model
- `confidence_metrics` and `prediction_metrics` — interval width and MWI scores, per model

All concatenation and `model`-column tagging happens inside `src/`; the notebook stays declarative.

In [10]:
report = compare_models(split_data, regressors, settings)
report.predictions.head()

Bootstrap: 100%|██████████| 200/200 [00:12<00:00, 16.39it/s]


,x,y_pred,y_true,model
0,-3.935980,-8.873145,-7.893230,random_forest
1,-1.354701,7.931102,6.002109,random_forest
2,3.924302,2.577703,-2.528021,random_forest
3,-7.003689,-18.342772,-17.681766,random_forest
4,5.243433,-0.657589,-1.662643,random_forest


## Visualize the comparison

`plot_intervals` renders one small-multiples panel per model: the data scatter,
the bootstrap confidence band, the conformal prediction band, and the regression line.

In [11]:
plot_intervals(data, report)

alt.HConcatChart(...)

## Pipeline evaluation

`plot_regression_metrics` facets by metric so each metric (RMSE, MAE, R²) gets its own y-axis —
comparing models across metrics on a shared scale would be misleading because the metrics live in
different units. Models sit on the x-axis within each facet, and each error bar shows the
bootstrap CI.

`plot_interval_metrics` facets by interval kind (confidence vs. prediction). Within each facet,
the metrics (width, MWI) are grouped on the x-axis and the models are color-coded — so the chart
compares confidence-against-confidence and prediction-against-prediction, not CI-against-PI.

In [12]:
plot_regression_metrics(report)

alt.FacetChart(...)

In [13]:
plot_interval_metrics(report)

alt.VConcatChart(...)

In [14]:
report.regression_metrics

,metric,lower,upper,model
0,rmse,2.141783,2.676804,random_forest
1,rmse,2.103177,2.611555,svm
2,mae,1.552589,1.975301,random_forest
3,mae,1.605875,1.959621,svm
4,r2,0.922186,0.957901,random_forest
5,r2,0.924409,0.960434,svm


In [15]:
report.confidence_metrics

,kind,metric,value,model
0,confidence,width,2.245033,random_forest
1,confidence,width,1.482228,svm


In [16]:
report.prediction_metrics

,kind,metric,value,model
0,prediction,width,10.574248,random_forest
1,prediction,width,11.024779,svm
2,prediction,mwi,12.759677,random_forest
3,prediction,mwi,12.694848,svm


## Tests

Each module under `src/` has a matching test module under `tests/`.
Run the full suite with:

```bash
uv run poe test
```

CI runs tests before building the book (`uv run poe ci`).